# Function Calling 🛠️🔌

Function Calling teaches you how to give an LLM hands so it can interact with the outside world.

In a software development, function calling is the bridge between static text generation and dynamic software execution. Instead of the LLM just guessing an answer, it can inspect a user's request, realize it needs real-time data or an action executed, and output a structured command telling your application which function to run and with what arguments.

## Phase 1: The Core Concept & Developer Analogy
**What is it?**
Function calling (sometimes called Tool Use) is an API feature where you provide the LLM with a manifest or schema of available functions in your backend codebase.


## Phase 2: The Function Calling Lifecycle (The Round-Trip Loop)
Unlike a standard single-turn API call, true function calling requires a multi-step communication loop between your application and the LLM provider:

**Step 1 (The Request):** You send the user prompt along with a JSON array describing your available functions (their names, descriptions, and expected parameters).

**Step 2 (The Tool Decision):** The LLM analyzes the prompt. If it needs external data, it pauses generation and returns a special tool-call response object containing the function name and parsed arguments.

**Step 3 (Local Execution):** Your backend code intercepts this response, executes your actual local function (e.g., making a real SQL query or third-party API call), and captures the output.

**Step 4 (The Synthesis):** You send the function's return value back to the LLM as a new message (role: "tool"). The LLM then reads that data and writes a natural language response back to the user.

## Phase 3: Python Code Implementation (OpenAI SDK)
Here is a clean, production-style implementation showing how to define a tool and handle the tool-call execution loop in Python

In [ ]:
import json
from openai import OpenAI

client = OpenAI()

# 1. Define a local Python function that your backend will execute
def get_current_server_status(service_name: str) -> str:
    """Mock function to check microservice health."""
    # In a real app, this would ping your Kubernetes cluster or database
    statuses = {
        "auth-service": "Healthy (CPU: 12%, Memory: 45%)",
        "billing-api": "Degraded (High latency detected)",
        "database": "Healthy (Replication lag: 0ms)"
    }
    return statuses.get(service_name.lower(), "Service not found.")

# 2. Map function name strings to actual executable Python functions
available_tools = {
    "get_current_server_status": get_current_server_status
}

def run_conversation(user_prompt: str):
    messages = [{"role": "user", "content": user_prompt}]
    
    # Define the tool schema for the LLM
    tools = [{
        "type": "function",
        "function": {
            "name": "get_current_server_status",
            "description": "Check the health and operational status of a backend microservice.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service_name": {
                        "type": "string",
                        "description": "The name of the service, e.g., auth-service, billing-api, database"
                    }
                },
                "required": ["service_name"],
            },
        }
    }]

    # Step 1: Send prompt and tool definitions to the LLM
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto", # Let the model decide if it needs to call a tool
    )
    
    response_message = response.choices[0].message
    
    # Step 2: Check if the model decided to call a function
    if response_message.tool_calls:
        print("🤖 LLM decided to call a function!")
        messages.append(response_message) # Append assistant's intent to history
        
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"Executing local function: {function_name} with args: {function_args}")
            
            # Step 3: Execute the local python function
            tool_to_call = available_tools[function_name]
            tool_output = tool_to_call(**function_args)
            
            # Step 4: Send the tool output back to the LLM
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": tool_output,
            })
            
        # Final API call to let the LLM synthesize the result into human text
        second_response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages
        )
        return second_response.choices[0].message.content
    
    return response_message.content

# Test the conversation loop
print("\n--- User Query 1 ---")
print(run_conversation("Hey, can you check the health of the billing-api for me?"))

## Phase 4: Why Function Calling is a Game Changer
**Eliminates Custom Regex Parsers:** You no longer need to write fragile prompt hacks trying to force an LLM to output custom command strings that you parse with split().

**Autonomous Agents:** Function calling is the foundational building block for AI Agents. By giving an LLM a suite of tools (web search, database queries, code execution sandboxes), it can iteratively solve complex multi-step problems on its own.

**Security & Control:** Because the LLM only outputs arguments while your backend code executes the actual function, you maintain complete programmatic control over database writes, API keys, and authorization boundaries.